# Train the "Orac" wake word (openWakeWord)

**Before running:** Runtime → Change runtime type → **T4 GPU**.
Then Runtime → **Run all** and come back in ~45–75 minutes.

Output: `orac_wake_models.zip` (auto-downloads at the end) containing
`orac.onnx`, `melspectrogram.onnx`, `embedding_model.onnx` — drop all
three into `public/models/` in the social-media-app repo.

In [ ]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install dependencies (ordered + guarded for current Colab images)
# Record Colab's preinstalled numpy so we can restore it after installs —
# stray pins from old audio packages can downgrade it and break the
# compiled stack (scipy/torch) with "numpy.dtype size changed" errors.
import numpy as np
open("/content/numpy_version.txt", "w").write(np.__version__)

!git clone https://github.com/dscripka/openWakeWord /content/openWakeWord
%cd /content/openWakeWord
!pip install -e .
%cd /content
# Pin piper-sample-generator to v2.0.0: the repo was later restructured into
# a package and no longer has the root generate_samples.py that
# openWakeWord's train.py imports.
!git clone https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!git -C /content/piper-sample-generator checkout v2.0.0
![ -f /content/piper-sample-generator/requirements.txt ] && pip install -q -r /content/piper-sample-generator/requirements.txt
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# Training extras (unpinned where safe — 2023 pins rot against new Colab)
!pip install mutagen torchinfo torchmetrics speechbrain==0.5.14 audiomentations acoustics onnx onnx2tf datasets pronouncing webrtcvad
!pip install -U torch-audiomentations
# deep-phonemizer provides 'dp' (adversarial negatives). --no-deps is
# REQUIRED: its old dependency pins downgrade numpy and break the compiled
# stack; everything it actually imports is already installed.
!pip install --no-deps deep-phonemizer
# piper-phonemize has no official Python 3.12 wheel; try community build too
!pip install piper-phonemize || pip install piper-phonemize-cross || echo "WARN: piper-phonemize unavailable — preflight will tell us"

# Restore Colab's exact numpy (no-deps so nothing else moves)
!pip install --force-reinstall --no-deps numpy==$(cat /content/numpy_version.txt)

# torchaudio 2.x removed info()/load() — patch torch_audiomentations' io.py
# to use soundfile (sitecustomize does NOT reach !python subprocesses on
# Colab — Debian's own sitecustomize shadows it — so patch the file itself).
import pathlib, glob as _g
_io = _g.glob("/usr/local/lib/python3*/dist-packages/torch_audiomentations/utils/io.py")[0]
p = pathlib.Path(_io)
src = p.read_text()
if "_oww_compat_patched" not in src:
    shim = '''
# _oww_compat_patched: torchaudio 2.x removed info()/load(); use soundfile
import soundfile as _sf
import torch as _torch
class _CompatAudioInfo:
    def __init__(self, frames, rate, ch):
        self.num_frames = frames; self.sample_rate = rate; self.num_channels = ch
        self.bits_per_sample = 16; self.encoding = "PCM_S"
def _compat_info(path, *a, **k):
    i = _sf.info(str(path))
    return _CompatAudioInfo(i.frames, i.samplerate, i.channels)
def _compat_load(path, *a, frame_offset=0, num_frames=-1, **k):
    data, sr = _sf.read(str(path), dtype="float32", always_2d=True,
                        start=frame_offset, frames=num_frames if num_frames and num_frames > 0 else -1)
    return _torch.from_numpy(data.T), sr
if not hasattr(torchaudio, "info"):
    torchaudio.info = _compat_info
if not hasattr(torchaudio, "load"):
    torchaudio.load = _compat_load
'''
    src = src.replace("import torchaudio", "import torchaudio" + shim, 1)
    p.write_text(src)

# openWakeWord's preprocessing models aren't bundled with the editable
# install — fetch them into the package (training needs them on disk).
import os
dst = "/content/openWakeWord/openwakeword/resources/models"
os.makedirs(dst, exist_ok=True)
try:
    import openwakeword.utils as _u
    _u.download_models(["melspectrogram", "embedding_model"])
except Exception as e:
    print("util download failed (falling back to direct):", e)
for name in ["melspectrogram.onnx", "embedding_model.onnx"]:
    pth = f"{dst}/{name}"
    if not os.path.exists(pth):
        os.system(f"wget -q -O {pth} https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{name}")
    assert os.path.exists(pth) and os.path.getsize(pth) > 0, f"failed to fetch {name}"
print("installed")

In [ ]:
# 3. Training config — "Orac" + "Hey Orac". ABSOLUTE paths throughout
# (train.py runs from /content/openWakeWord, so relative paths break).
# Start from the project's own example config so every required key exists,
# then overlay our values.
import yaml, glob

ours = {
    "target_phrase": ["orac", "hey orac"],
    "custom_negative_phrases": ["oracle", "a rack", "or act", "okay"],
    "model_name": "orac",
    # v2 recipe — the 6k/30k/neg-weight-1500 first run produced a model that
    # only fired for ONE voice (speaker-overfit): 9-voice test showed 0.711
    # for a single en-GB TTS voice and ~0.00 for every other voice. More
    # positives, gentler negative weighting and higher recall targets are
    # what production openWakeWord models use.
    "n_samples": 30000,
    "n_samples_val": 2000,
    "steps": 50000,
    "max_negative_weight": 500,
    "target_accuracy": 0.85,
    "target_recall": 0.75,
    "target_false_positives_per_hour": 0.2,
    # 2s window → 16 embedding frames; matches the app's runtime engine
    "total_length": 32000,
    "background_paths": ["/content/audioset_16k", "/content/fma"],
    "rir_paths": ["/content/rirs"],
    "false_positive_validation_data_path": "/content/validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 100, "positive": 100},
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": "/content/orac_model",
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "augmentation_rounds": 2,
    "layer_size": 32,
}

template = {}
for p in glob.glob("/content/openWakeWord/examples/*.yml") + glob.glob("/content/openWakeWord/**/*.yml", recursive=True):
    try:
        d = yaml.safe_load(open(p)) or {}
        if isinstance(d, dict) and ("n_samples" in d or "target_phrase" in d):
            template = d
            print("template:", p)
            break
    except Exception:
        pass

cfg = dict(template)
cfg.update(ours)
yaml.safe_dump(cfg, open("/content/orac.yaml", "w"), sort_keys=False)
print("config written —", len(cfg), "keys")

In [ ]:
# 4. Download negative/validation feature sets + create background audio + RIRs
%cd /content
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Background noise + room-impulse-responses via the stdlib `wave` module —
# deliberately NO numpy/scipy (immune to binary-compat state) and no
# external downloads (URLs rot).
import wave, struct, random, os
random.seed(42)
for d in ["/content/fma", "/content/audioset_16k"]:
    os.makedirs(d, exist_ok=True)
    if not os.listdir(d):
        for i in range(60):
            with wave.open(f"{d}/noise_{i}.wav", "w") as w:
                w.setnchannels(1)
                w.setsampwidth(2)
                w.setframerate(16000)
                amp = random.uniform(300, 2500)
                prev = 0.0
                frames = bytearray()
                for _ in range(16000 * 10):  # 10s, low-pass shaped noise
                    prev = 0.85 * prev + random.uniform(-1, 1)
                    v = int(prev * amp)
                    frames += struct.pack("<h", max(-32768, min(32767, v)))
                w.writeframes(bytes(frames))

# RIRs: decaying noise bursts ≈ room reverb kernels
rir_dir = "/content/rirs"
os.makedirs(rir_dir, exist_ok=True)
random.seed(7)
if not os.listdir(rir_dir):
    for i in range(20):
        with wave.open(f"{rir_dir}/rir_{i}.wav", "w") as w:
            w.setnchannels(1)
            w.setsampwidth(2)
            w.setframerate(16000)
            n = int(16000 * random.uniform(0.15, 0.4))
            decay = random.uniform(20, 60)
            frames = bytearray(struct.pack("<h", 28000))  # direct impulse
            for t in range(1, n):
                v = random.uniform(-1, 1) * 28000 * (2.718 ** (-decay * t / 16000))
                frames += struct.pack("<h", int(max(-32768, min(32767, v))))
            w.writeframes(bytes(frames))
print("datasets ready")

# Validate every generated wav — a single truncated header crashes the
# augment stage hours later ("Format not recognised" on a random batch).
import soundfile as _sf
_bad = 0
for _d in ["/content/audioset_16k", "/content/fma", "/content/rirs"]:
    if not os.path.isdir(_d): continue
    for _f in os.listdir(_d):
        _p = os.path.join(_d, _f)
        try:
            assert _sf.info(_p).frames > 0
        except Exception:
            os.remove(_p)
            _bad += 1
assert _bad == 0, f"{_bad} unreadable wav(s) removed — re-run this cell to regenerate"
print("all noise/RIR files validated")

In [ ]:
# 4.5 PREFLIGHT — verify every import, file, and config key the training
# needs. Fails fast with a named problem instead of a mid-train error.
import importlib, sys, os, re, yaml
sys.path.append("/content/piper-sample-generator")
problems = []
for m in ["pronouncing", "webrtcvad", "torch_audiomentations", "speechbrain",
          "datasets", "mutagen", "acoustics", "audiomentations", "onnx",
          "torchmetrics", "generate_samples", "dp.phonemizer"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        problems.append(f"import {m}: {type(e).__name__}: {e}")
for f in ["/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
          "/content/validation_set_features.npy",
          "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",
          "/content/openWakeWord/openwakeword/resources/models/melspectrogram.onnx",
          "/content/openWakeWord/openwakeword/resources/models/embedding_model.onnx",
          "/content/orac.yaml"]:
    if not os.path.exists(f):
        problems.append(f"missing file: {f}")
for d in ["/content/fma", "/content/rirs"]:
    if not (os.path.isdir(d) and os.listdir(d)):
        problems.append(f"no clips in {d}")
# every config["..."] train.py reads must exist in our yaml
cfg = yaml.safe_load(open("/content/orac.yaml"))
req = set(re.findall(r'config\[[\'"]([A-Za-z_0-9]+)[\'"]\]',
                     open("/content/openWakeWord/openwakeword/train.py").read()))
missing_keys = sorted(req - set(cfg.keys()))
if missing_keys:
    problems.append(f"config keys train.py reads but yaml lacks: {missing_keys}")
import numpy
print("numpy:", numpy.__version__)
import scipy  # binary-compat canary
print("scipy:", scipy.__version__, "(imports cleanly — no numpy mismatch)")
assert not problems, "PREFLIGHT FAILED — fix these before training:\n" + "\n".join(problems)
print("\nPREFLIGHT OK — run the training cell")

In [ ]:
# 5. Generate synthetic samples, augment, train (the long cell — ~30-60 min)
# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD: PyTorch 2.6 flipped torch.load to
# weights_only=True; the Piper checkpoint and other pipeline artifacts are
# full pickled models from trusted release files.
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
!sed -i 's/torch\.load(model_path)/torch.load(model_path, weights_only=False)/' /content/piper-sample-generator/generate_samples.py
%cd /content/openWakeWord
!python openwakeword/train.py --training_config /content/orac.yaml --generate_clips
!python openwakeword/train.py --training_config /content/orac.yaml --augment_clips
!python openwakeword/train.py --training_config /content/orac.yaml --train_model
%cd /content

In [ ]:
# 6. Bundle the three runtime models and download
import shutil, glob, os
import openwakeword
%cd /content
os.makedirs("bundle", exist_ok=True)
# the trained classifier
trained = glob.glob("/content/orac_model/**/orac.onnx", recursive=True) + glob.glob("/content/**/orac.onnx", recursive=True)
assert trained, "orac.onnx not found — check the training cell output above"
# merge external weights (.onnx.data) if the exporter split them — the
# browser needs a single self-contained file
import onnx
from onnx.external_data_helper import convert_model_from_external_data
_m = onnx.load(trained[0])
convert_model_from_external_data(_m)
onnx.save(_m, "bundle/orac.onnx")
assert os.path.getsize("bundle/orac.onnx") > 100_000, "orac.onnx looks weightless — external data missing?"
# the shared preprocessing models from the installed package (download if needed)
try:
    from openwakeword import utils as oww_utils
    if hasattr(oww_utils, "download_models"):
        oww_utils.download_models(["melspectrogram", "embedding_model"])
except Exception as e:
    print("download_models skipped:", e)
pkg = os.path.dirname(openwakeword.__file__)
for name in ["melspectrogram.onnx", "embedding_model.onnx"]:
    found = glob.glob(os.path.join(pkg, "resources", "models", name)) or glob.glob(f"/content/**/{name}", recursive=True)
    assert found, f"{name} not found"
    shutil.copy(found[0], f"bundle/{name}")
shutil.make_archive("orac_wake_models", "zip", "bundle")
print("Bundle contents:", os.listdir("bundle"))
from google.colab import files
files.download("/content/orac_wake_models.zip")

## Done

Unzip `orac_wake_models.zip` and put all three `.onnx` files into
`public/models/` in the social-media-app repo, then commit + push.
The app switches engines automatically.

If any cell failed, see `scripts/wake-training/README.md` — the upstream
notebook is the source of truth and this config pastes straight into it.